# AI Eye Image Analysis & Diabetes Risk Screening

This notebook walks through:
1. **Scientific Task Separation**: Retinal fundus diabetic retinopathy grading vs. experimental iris segmentation CV vs. separate tabular Pima Indians Diabetes prediction (and why direct blood glucose in mg/dL is NOT estimated from eye images).
2. **Dataset Inspection & Quality Verification**: Class distribution, resolution checks, deduplication, and patient-disjoint splits.
3. **Preprocessing & Augmentation Preview**: Border cropping, LAB CLAHE enhancement, and augmentation transforms.
4. **Model Training & Independent Test Evaluation**: Accuracy, macro/weighted F1, sensitivity, false-negative screening counts, confusion matrix, and ROC-AUC.

In [ ]:
import os, sys, json
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, ROOT_DIR)

with open(os.path.join(ROOT_DIR, 'data', 'retinal_fundus', 'dataset_summary.json')) as f:
    ret_summary = json.load(f)
print(json.dumps(ret_summary, indent=2))

In [ ]:
# Visualize class distribution across Train, Validation, and Held-out Test splits
train_df = pd.read_csv(os.path.join(ROOT_DIR, 'data', 'retinal_fundus', 'train.csv'))
val_df = pd.read_csv(os.path.join(ROOT_DIR, 'data', 'retinal_fundus', 'val.csv'))
test_df = pd.read_csv(os.path.join(ROOT_DIR, 'data', 'retinal_fundus', 'test.csv'))

dist_df = pd.DataFrame({
    'Train': train_df['label_name'].value_counts(),
    'Val': val_df['label_name'].value_counts(),
    'Test': test_df['label_name'].value_counts()
}).fillna(0).astype(int)
print(dist_df)
dist_df.plot(kind='bar', figsize=(9, 4), title='Retinal DR Class Distribution Across Splits')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Preview retinal preprocessing (Border crop + LAB CLAHE)
import numpy as np
from src.eye_analysis.preprocessing import preprocess_fundus_rgb

sample_row = train_df.iloc[0]
orig = Image.open(os.path.join(ROOT_DIR, sample_row['image_path'])).convert('RGB')
proc = preprocess_fundus_rgb(np.asarray(orig), target_size=224, apply_clahe=True)

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(orig); ax[0].set_title(f"Raw ({sample_row['label_name']})"); ax[0].axis('off')
ax[1].imshow(proc); ax[1].set_title('Preprocessed (Crop + CLAHE)'); ax[1].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# Review saved held-out test set evaluation report & plots
with open(os.path.join(ROOT_DIR, 'models', 'evaluation_report.json')) as f:
    eval_rep = json.load(f)
print('Overall Test Metrics:', json.dumps(eval_rep['overall_metrics'], indent=2))
print('Screening Sensitivity Analysis:', json.dumps(eval_rep['screening_sensitivity_analysis'], indent=2))

for plot_name in ['training_history.png', 'confusion_matrix.png', 'roc_curves.png']:
    p = os.path.join(ROOT_DIR, 'models', plot_name)
    if os.path.exists(p):
        display(Image.open(p))

In [ ]:
# Separate Tabular Clinical Risk Model (Pima Indians Diabetes Dataset)
with open(os.path.join(ROOT_DIR, 'models', 'pima_diabetes_eval.json')) as f:
    pima_rep = json.load(f)
print(json.dumps(pima_rep, indent=2))